# 0. import libraries

In [1]:
# @title Colab Setup Environment

try:
    import google.colab

    # Remove existing directory to ensure clean clone on re-run
    !rm -rf repository/circuit-tracer

    !mkdir -p repository && cd repository && \
     git clone https://github.com/safety-research/circuit-tracer && \
     curl -LsSf https://astral.sh/uv/install.sh | sh && \
     uv pip install -e circuit-tracer/

    import sys
    from huggingface_hub import notebook_login

    sys.path.append("repository/circuit-tracer")
    sys.path.append("repository/circuit-tracer/demos")
    notebook_login(new_session=False)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

In [2]:
# ── 1. HuggingFace model for generation ──────────────────────────────────────
from transformers import AutoTokenizer
from circuit_tracer import ReplacementModel, attribute
from circuit_tracer.utils import create_graph_files
import torch
import torch.nn.functional as F

tokenizer = AutoTokenizer.from_pretrained("google/gemma-2-2b")

# ── 2. circuit_tracer model for attribution (replaces HF model entirely) ─────
model = ReplacementModel.from_pretrained(
    "google/gemma-2-2b", "gemma", dtype=torch.bfloat16, backend="transformerlens"
)

Fetching 26 files:   0%|          | 0/26 [00:00<?, ?it/s]

`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Loaded pretrained model google/gemma-2-2b into HookedTransformer


# 1. getting attributes from each token step
based on attribute_demo  
**can skip to step 2, features are saved in** `demos\graph_files`

In [3]:
max_n_logits = 10  # How many logits to attribute from, max. We attribute to min(max_n_logits, n_logits_to_reach_desired_log_prob); see below for the latter
desired_logit_prob = 0.95  # Attribution will attribute from the minimum number of logits needed to reach this probability mass (or max_n_logits, whichever is lower)
max_feature_nodes = 2048   # Only attribute from this number of feature nodes, max. Lower is faster, but you will lose more of the graph. None means no limit.
batch_size = 256  # Batch size when attributing
offload = (
    "disk" if IN_COLAB else "disk"
)  # Offload various parts of the model during attribution to save memory. Can be 'disk', 'cpu', or None (keep on GPU)
verbose = True  # Whether to display a tqdm progress bar and timing report

In [4]:
# ── 3. Generation loop (same as before, but using ReplacementModel) ───────────
base_prompt = "A rhyming couplet:\nHe stared into the orange sun,\n"
input_ids = tokenizer(base_prompt, return_tensors="pt")["input_ids"]
STOP_IDS = {108, 235265, tokenizer.eos_token_id}
token_trace = []

for step in range(20):
    with torch.no_grad():
      out = model(input_ids)

    logits_last = out[0, -1, :].float()  # out is already the logits tensor
    probs = F.softmax(logits_last, dim=-1)
    top_probs, top_ids = torch.topk(probs, 10)
    next_id = top_ids[0].item()

    token_trace.append({
        "step": step,
        "token_id": next_id,
        "token_str": tokenizer.decode([next_id]),
        "chosen_prob": top_probs[0].item(),
        "chosen_logit": logits_last[next_id].item(),
        "top10": [{"token": tokenizer.decode([tid.item()]), "prob": p.item()}
                  for tid, p in zip(top_ids, top_probs)],
    })

    if next_id in STOP_IDS:
        break

    input_ids = torch.cat(
        [input_ids, torch.tensor([[next_id]])], dim=1
    )

print("Generated:", "".join(t["token_str"] for t in token_trace if t["token_id"] not in STOP_IDS))

# ── 4. Attribution loop — token_trace flows straight in ──────────────────────
for i, token in enumerate(token_trace):
    if token["token_id"] in STOP_IDS:
        break

    tokens_so_far = "".join(t["token_str"] for t in token_trace[:i])
    prompt_at_step = base_prompt + tokens_so_far

    graph = attribute(
        prompt=prompt_at_step,
        model=model,
        max_n_logits=max_n_logits,
        desired_logit_prob=desired_logit_prob,
        max_feature_nodes=max_feature_nodes,   # ← missing
        batch_size=batch_size,
        offload=offload,                        # ← use the variable, not hardcoded "cpu"
        verbose=verbose,
    )

    slug = f"step-{i:02d}-{token['token_str'].strip().replace(' ', '_')}"
    create_graph_files(
        graph_or_path=graph,
        slug=slug,
        output_path="./graph_files/gemma-2-2b-2/",
        node_threshold=0.8,
        edge_threshold=0.98,
    )
    print(f"✓ step {i:02d} → '{token['token_str']}'  saved as '{slug}'")

Phase 0: Precomputing activations and vectors


Generated: And saw the world in a new way


Precomputation completed in 0.91s
Found 13259 active features
Phase 1: Running forward pass
Forward pass completed in 13.35s
Phase 2: Building input vectors
Using 10 salient logits with cumulative probability 0.5664
Will include 2048 of 13259 feature nodes
Input vectors built in 3.36s
Phase 3: Computing logit attributions
c:\Users\sirichada\Github\circuit-tracer\.venv\Lib\site-packages\circuit_tracer\attribution\context_transformerlens.py:223: UserWarning: Full backward hook is firing when gradients are computed with respect to module outputs since no inputs require gradients. See https://docs.pytorch.org/docs/main/generated/torch.nn.Module.html#torch.nn.Module.register_full_backward_hook for more details.
  self._resid_activations[last_layer].backward(
Logit attributions completed in 4.37s
Phase 4: Computing feature attributions
Feature influence computation: 100%|██████████| 2048/2048 [00:28<00:00, 70.79it/s]
Feature attributions completed in 28.93s
Attribution completed in 54.14s
Ph

✓ step 00 → 'And'  saved as 'step-00-And'


Precomputation completed in 10.51s
Found 14075 active features
Phase 1: Running forward pass
Forward pass completed in 20.64s
Phase 2: Building input vectors
Using 10 salient logits with cumulative probability 0.5000
Will include 2048 of 14075 feature nodes
Input vectors built in 3.02s
Phase 3: Computing logit attributions
Logit attributions completed in 4.68s
Phase 4: Computing feature attributions
Feature influence computation: 100%|██████████| 2048/2048 [00:32<00:00, 62.12it/s]
Feature attributions completed in 32.97s
Attribution completed in 77.65s
Phase 0: Precomputing activations and vectors


...

✓ step 05 → ' a'  saved as 'step-05-a'


Precomputation completed in 1.42s
Found 18536 active features
Phase 1: Running forward pass
Forward pass completed in 23.09s
Phase 2: Building input vectors
Using 10 salient logits with cumulative probability 0.4766
Will include 2048 of 18536 feature nodes
Input vectors built in 2.25s
Phase 3: Computing logit attributions
Logit attributions completed in 6.02s
Phase 4: Computing feature attributions
Feature influence computation: 100%|██████████| 2048/2048 [00:44<00:00, 46.52it/s]
Feature attributions completed in 44.03s
Attribution completed in 81.05s
Phase 0: Precomputing activations and vectors


✓ step 06 → ' new'  saved as 'step-06-new'


Precomputation completed in 1.49s
Found 19475 active features
Phase 1: Running forward pass
Forward pass completed in 24.27s
Phase 2: Building input vectors
Using 10 salient logits with cumulative probability 0.8789
Will include 2048 of 19475 feature nodes
Input vectors built in 2.92s
Phase 3: Computing logit attributions
Logit attributions completed in 6.33s
Phase 4: Computing feature attributions
Feature influence computation: 100%|██████████| 2048/2048 [00:49<00:00, 41.52it/s]
Feature attributions completed in 49.33s
Attribution completed in 88.26s


✓ step 07 → ' way'  saved as 'step-07-way'


In [10]:
import json
from pathlib import Path

total = sum(
    sum(1 for node in json.loads(p.read_text())["nodes"]
        if not node["is_target_logit"] and node["influence"] is not None)
    for p in sorted(Path("./graph_files/gemma-2-2b-2").glob("step-*.json"))
)
print(f"{total} nodes → ~{total * 0.1 / 60:.1f} minutes")

8147 nodes → ~13.6 minutes


In [12]:
# extracting clerp descriptions and adding to graph files

import json, time, requests
from pathlib import Path

def fetch_clerp(layer, feature):
    url = f"https://www.neuronpedia.org/api/feature/gemma-2-2b/{layer}-gemmascope-transcoder-16k/{feature}"
    r = requests.get(url)
    if r.ok:
        data = r.json()
        descs = data.get("explanations", [])
        if descs:
            return descs[0].get("description", "")
    return ""

for json_path in sorted(Path("./graph_files/gemma-2-2b-2").glob("step-*.json")):
    data = json.loads(json_path.read_text())
    for node in data['nodes']:
        if node['is_target_logit'] or node['influence'] is None:
            continue
        parts = node['node_id'].split('_')
        feat = parts[1]
        layer = node['layer']
        node['clerp'] = fetch_clerp(layer, feat)
        time.sleep(0.1)  # be nice to the API
    json_path.write_text(json.dumps(data))
    print(f"done: {json_path.name}")

done: step-00-And.json
done: step-01-saw.json
done: step-02-the.json
done: step-03-world.json
done: step-04-in.json
done: step-05-a.json
done: step-06-new.json
done: step-07-way.json


# 2. attribute graph visualization

In [13]:
import json
from pathlib import Path

graph_dir = Path("./graph_files/gemma-2-2b-2")

for json_path in sorted(graph_dir.glob("step-*.json")):
    data = json.loads(json_path.read_text())
    
    # collect all unique layers used by transcoder nodes
    layers = sorted(set(
        int(node["layer"])
        for node in data["nodes"]
        if not node["is_target_logit"] and node.get("feature_type") == "cross layer transcoder"
    ))
    
    transcoder_list = [f"gemma-2-2b/{l}-gemmascope-transcoder-16k" for l in layers]
    data["metadata"]["transcoder_list"] = transcoder_list
    
    json_path.write_text(json.dumps(data))
    print(f"{json_path.name}: {len(layers)} layers → {transcoder_list[:3]}...")

# also patch the manifest
manifest_path = graph_dir / "graph-metadata.json"
manifest = json.loads(manifest_path.read_text())
for entry in manifest["graphs"]:
    slug = entry["slug"]
    step_path = graph_dir / f"{slug}.json"
    if step_path.exists():
        step_data = json.loads(step_path.read_text())
        entry["transcoder_list"] = step_data["metadata"]["transcoder_list"]

manifest_path.write_text(json.dumps(manifest, indent=2))
print("manifest patched")

step-00-And.json: 26 layers → ['gemma-2-2b/0-gemmascope-transcoder-16k', 'gemma-2-2b/1-gemmascope-transcoder-16k', 'gemma-2-2b/2-gemmascope-transcoder-16k']...
step-01-saw.json: 26 layers → ['gemma-2-2b/0-gemmascope-transcoder-16k', 'gemma-2-2b/1-gemmascope-transcoder-16k', 'gemma-2-2b/2-gemmascope-transcoder-16k']...
step-02-the.json: 26 layers → ['gemma-2-2b/0-gemmascope-transcoder-16k', 'gemma-2-2b/1-gemmascope-transcoder-16k', 'gemma-2-2b/2-gemmascope-transcoder-16k']...
step-03-world.json: 26 layers → ['gemma-2-2b/0-gemmascope-transcoder-16k', 'gemma-2-2b/1-gemmascope-transcoder-16k', 'gemma-2-2b/2-gemmascope-transcoder-16k']...
step-04-in.json: 26 layers → ['gemma-2-2b/0-gemmascope-transcoder-16k', 'gemma-2-2b/1-gemmascope-transcoder-16k', 'gemma-2-2b/2-gemmascope-transcoder-16k']...
step-05-a.json: 26 layers → ['gemma-2-2b/0-gemmascope-transcoder-16k', 'gemma-2-2b/1-gemmascope-transcoder-16k', 'gemma-2-2b/2-gemmascope-transcoder-16k']...
step-06-new.json: 26 layers → ['gemma-2-2

In [14]:
import json
from pathlib import Path

data = {"graphs": []}  # ← Add this line to initialize

for json_path in sorted(Path("./graph_files/gemma-2-2b-2").glob("step-*.json")):
    graph_data = json.loads(json_path.read_text())
    data["graphs"].append(graph_data["metadata"])

Path("./graph_files/gemma-2-2b-2/graph-metadata.json").write_text(json.dumps(data, indent=2))
print("done")

done


In [1]:
from circuit_tracer.frontend.local_server import serve
from IPython.display import IFrame

port = 8047
server = serve(data_dir="./graph_files/gemma-2-2b-2/", port=port)

print(f"http://localhost:{port}/index.html")
display(IFrame(src=f"http://localhost:{port}/index.html", width="100%", height="800px"))

http://localhost:8047/index.html


In [2]:
server.shutdown()

AttributeError: 'Server' object has no attribute 'shutdown'

## 2.1 feature analysis based on circuit

In [17]:
import json
from pathlib import Path

def show_top_nodes(json_path, n=10):
    data = json.loads(Path(json_path).read_text())
    slug = data['metadata']['slug']
    
    # filter out logit nodes, sort by influence
    nodes = [n for n in data['nodes'] if not n['is_target_logit'] and n['influence'] is not None]
    nodes = sorted(nodes, key=lambda x: x['influence'], reverse=True)[:n]
    
    print(f"\n=== {slug} ===")
    for node in nodes:
        layer = node['layer']
        node_id_parts = node['node_id'].split('_')
        feat = node_id_parts[1]
        influence = node['influence']
        url = f"https://www.neuronpedia.org/gemma-2-2b/{layer}-gemmascope-transcoder-16k/{feat}"
        print(f"  L{layer}F{feat:>6}  influence={influence:.4f}  {url}")

for f in sorted(Path("./graph_files/gemma-2-2b-2").glob("step-*.json")):
    show_top_nodes(f)


=== step-00-And ===
  L21F 11124  influence=0.8001  https://www.neuronpedia.org/gemma-2-2b/21-gemmascope-transcoder-16k/11124
  L3F  7499  influence=0.7999  https://www.neuronpedia.org/gemma-2-2b/3-gemmascope-transcoder-16k/7499
  L13F    13  influence=0.7997  https://www.neuronpedia.org/gemma-2-2b/13-gemmascope-transcoder-16k/13
  L20F    20  influence=0.7994  https://www.neuronpedia.org/gemma-2-2b/20-gemmascope-transcoder-16k/20
  L7F 14826  influence=0.7992  https://www.neuronpedia.org/gemma-2-2b/7-gemmascope-transcoder-16k/14826
  L2F  1984  influence=0.7990  https://www.neuronpedia.org/gemma-2-2b/2-gemmascope-transcoder-16k/1984
  L10F  7747  influence=0.7988  https://www.neuronpedia.org/gemma-2-2b/10-gemmascope-transcoder-16k/7747
  L0F 14284  influence=0.7985  https://www.neuronpedia.org/gemma-2-2b/0-gemmascope-transcoder-16k/14284
  L12F 12564  influence=0.7983  https://www.neuronpedia.org/gemma-2-2b/12-gemmascope-transcoder-16k/12564
  L2F     2  influence=0.7981  https://www

In [18]:
import json
from pathlib import Path
from collections import defaultdict

position_features = defaultdict(list)

for json_path in sorted(Path("./graph_files/gemma-2-2b-2").glob("step-*.json")):
    data = json.loads(json_path.read_text())
    for node in data['nodes']:
        if node['is_target_logit'] or node['influence'] is None:
            continue
        token = data['metadata']['prompt_tokens'][node['ctx_idx']]
        if token == '\n':
            parts = node['node_id'].split('_')
            key = f"L{node['layer']}F{parts[1]}"
            position_features[key].append((json_path.stem, node['influence'], node['clerp']))

# show features appearing in 3+ steps, sorted by occurrence count
for feat, occurrences in sorted(position_features.items(), key=lambda x: -len(x[1])):
    if len(occurrences) >= 3:
        clerp = occurrences[0][2] or "no description"
        print(f"{feat}  ({len(occurrences)}/9 steps)  clerp: {clerp}")
        for step, influence, _ in occurrences:
            print(f"    {step}  influence={influence:.4f}")

L0F3850  (16/9 steps)  clerp: punctuation marks
    step-00-And  influence=0.5000
    step-00-And  influence=0.4204
    step-01-saw  influence=0.4779
    step-01-saw  influence=0.5626
    step-02-the  influence=0.5354
    step-02-the  influence=0.5227
    step-03-world  influence=0.5657
    step-03-world  influence=0.5953
    step-04-in  influence=0.5832
    step-04-in  influence=0.4918
    step-05-a  influence=0.5756
    step-05-a  influence=0.5016
    step-06-new  influence=0.5833
    step-06-new  influence=0.5260
    step-07-way  influence=0.5304
    step-07-way  influence=0.5444
L0F7894  (16/9 steps)  clerp:  line breaks that indicate a paragraph or section beginning
    step-00-And  influence=0.4092
    step-00-And  influence=0.3591
    step-01-saw  influence=0.3988
    step-01-saw  influence=0.5397
    step-02-the  influence=0.4701
    step-02-the  influence=0.6009
    step-03-world  influence=0.4235
    step-03-world  influence=0.5927
    step-04-in  influence=0.4383
    step-04

**Most promising (identified by Claude Sonnet 4.6):**

L2F629 — "marks verses, choruses, or segments of a musical composition" — active on \n across all 9 steps, consistently  
L8F13615 — "words and phrases related to poetry or creative writing with a bias toward personal anecdotes" — 17/9 steps  
L4F4619 — "lyrics of a specific song, possibly about love and uncertainty" — 17/9 steps  
L5F6651 — "sad and inquisitive poetry" — 9/9 steps, consistent influence ~0.6-0.78  
L8F12808 — "lines of poetry, especially those that express questioning, prayer, or lament" — 9/9 steps  
L13F4053 — "lines of rap lyrics" — 12/9 steps, high influence  
L3F14659 — "the language of rap lyrics" — 18/9 steps  
L14F15021 — "phrases that begin a line of poetry" — this one is particularly interesting given the context  
L9F11782 — "instances of the word 'poem' or descriptive phrases about things, often with a sense of place" — 5/9 steps  
L8F13396 — "mentions of poetic or musical art forms" — 5/9 steps, hits 0.8000 influence at step-07

In [12]:
# Now do the same for the other graph and compare overlaps

import json
from pathlib import Path
from collections import defaultdict

def get_position_features(graph_dir, min_steps=3):
    position_features = defaultdict(list)
    for json_path in sorted(Path(graph_dir).glob("step-*.json")):
        data = json.loads(json_path.read_text())
        for node in data['nodes']:
            if node['is_target_logit'] or node['influence'] is None:
                continue
            token = data['metadata']['prompt_tokens'][node['ctx_idx']]
            if token == '\n':
                parts = node['node_id'].split('_')
                key = f"L{node['layer']}F{parts[1]}"
                position_features[key].append((json_path.stem, node['influence'], node.get('clerp', '')))
    return {k: v for k, v in position_features.items() if len(v) >= min_steps}

carrot = get_position_features("./graph_files/gemma-2-2b-1")
sun    = get_position_features("./graph_files/gemma-2-2b-2")

overlapping = set(carrot.keys()) & set(sun.keys())

for feat in sorted(overlapping):
    clerp = carrot[feat][0][2] or sun[feat][0][2] or "no description"
    print(f"{feat}  carrot={len(carrot[feat])} steps  sun={len(sun[feat])} steps  clerp: {clerp}")

L0F1365  carrot=10 steps  sun=7 steps  clerp:  text appearing at the beginning of an article, followed by some sort of extra blank lines or spacing
L0F14055  carrot=18 steps  sun=11 steps  clerp:  seemingly random formatting and symbols
L0F14751  carrot=18 steps  sun=16 steps  clerp: code blocks, especially related to website development (HTML, CSS, Javascript)
L0F16273  carrot=18 steps  sun=13 steps  clerp:  a mix of formatting codes and the word "the"
L0F3259  carrot=15 steps  sun=3 steps  clerp: indented or spaced areas in documents
L0F3850  carrot=18 steps  sun=16 steps  clerp: punctuation marks
L0F7894  carrot=18 steps  sun=16 steps  clerp:  line breaks that indicate a paragraph or section beginning
L0F9818  carrot=13 steps  sun=8 steps  clerp: indented code blocks or code-related formatting.
L10F10  carrot=18 steps  sun=10 steps  clerp:  various topics including consumer electronics, law, and online forums.
L10F1056  carrot=7 steps  sun=6 steps  clerp: slang used in rap lyrics wi

# 3. intervention
based on intervention_demo

In [1]:
from collections import namedtuple
from functools import partial

import torch 

from circuit_tracer import ReplacementModel

# display functions
from circuit_tracer.utils.demo_utils import display_topk_token_predictions, display_generations_comparison

backend = 'transformerlens'  # change to 'nnsight' for the nnsight backend!
model = ReplacementModel.from_pretrained("google/gemma-2-2b", "gemma", dtype=torch.bfloat16, backend=backend)


Fetching 26 files:   0%|          | 0/26 [00:00<?, ?it/s]

`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Loaded pretrained model google/gemma-2-2b into HookedTransformer


In [2]:
Feature = namedtuple('Feature', ['layer', 'pos', 'feature_idx'])

# a display function that needs the model's tokenizer
display_topk_token_predictions = partial(display_topk_token_predictions, tokenizer=model.tokenizer)

In [ ]:
# L8F13615  = words and phrases related to poetry or creative writing
supernode_features = [
    Feature(layer=8,pos=-1,feature_idx=13615),
]

intervention_tuples = [(*supernode_feature, 0.0) for supernode_feature in supernode_features]

In [4]:
s = "A rhyming couplet:\nHe stared into the orange sun,\n"

In [5]:
with torch.inference_mode():
    original_logits, _  = model.feature_intervention(s, [])
    new_logits, _ = model.feature_intervention(s, intervention_tuples)

display_topk_token_predictions(s, original_logits, new_logits)

Token,Probability,Distribution
And,0.186,18.6%
The,0.077,7.7%
He,0.053,5.3%
A,0.047,4.7%
As,0.047,4.7%
Token,Probability,Distribution
And,0.192,19.2%
The,0.071,7.1%
He,0.055,5.5%
As,0.049,4.9%


In [10]:
from circuit_tracer.utils.demo_utils import display_generations_comparison

s = "A rhyming couplet:\nHe stared into the orange sun,\n"

# suppress overlapping poetry features from both poems
intervention_tuples = [
    (2,  -1, 629,   0.0),  # L2F629   - marks verses, choruses, or segments of a musical composition
    (3,  -1, 14659, 0.0),  # L3F14659 - the language of rap lyrics
    (4,  -1, 4619,  0.0),  # L4F4619  - lyrics of a specific song, possibly about love and uncertainty
    (8,  -1, 13615, 0.0),  # L8F13615 - words and phrases related to poetry or creative writing
    (8,  -1, 12808, 0.0),  # L8F12808 - lines of poetry, especially those that express questioning, prayer, or lament
    (13, -1, 4053,  0.0),  # L13F4053 - lines of rap lyrics
    (14, -1, 15021, 0.0),  # L14F15021 - phrases that begin a line of poetry
    (3,  -1, 1614,  0.0),  # L3F1614  -  tokens in song lyrics or poems relating to suffering, hardship
]

pre = [model.feature_intervention_generate(s, [], do_sample=False, verbose=False)[0]]
post = [model.feature_intervention_generate(s, intervention_tuples, do_sample=False, verbose=False)[0]]

display_generations_comparison(s, pre, post)